In [1]:
import lightning as L
from lightning.pytorch.loggers import CSVLogger
from lightning.pytorch.callbacks import DeviceStatsMonitor

In [2]:
from BNSReg.core.config import BNSDataModuleRegressionConfig, S4DModelConfig
from BNSReg.dataloader.regression_loader import LitBNSDataRegression
from BNSReg.tasks.parameter_estimation.model_s4d_mse import LitModelS4DMSE

In [3]:
data_cfg = BNSDataModuleRegressionConfig(
    target_variables=('chirp_mass',),
    observed_variables=('snr',),

    # strain variables
    strain_frequency=512,
    strain_duration=64,
    coalescence_time=63,
    window_begin=0,
    window_end=40,
    downsample_factor=2,

    # I/O variables
    strain_precision='torch.float32',
    variables_precision='torch.float32',
    rdcc_nbytes=64 * 1024**2,
    rdcc_nslots=10_007,
    rdcc_w0=0.75,

    # dataloader variables
    train_file='/n/holystore01/LABS/iaifi_lab/Lab/kyoon/DATA/bns_snr_15_25_uniform_512Hz_10K/train/sig_combined_train.h5',
    test_file='/n/holystore01/LABS/iaifi_lab/Lab/kyoon/DATA/bns_snr_15_25_uniform_512Hz_10K/test/sig_combined_test.h5',
    val_file='/n/holystore01/LABS/iaifi_lab/Lab/kyoon/DATA/bns_snr_15_25_uniform_512Hz_10K/val/sig_combined_val.h5',
    train_batch_size=1,
    val_batch_size=1,
    test_batch_size=1,
    num_workers=0,
    shuffle=False,
    prefetch_factor=None,
    persistent_workers=False,
)

In [4]:
model_cfg = S4DModelConfig(
    d_input=2,
    d_output=1,
    d_model=2,
    d_state=2,
    n_layers=1,
    dropout=0.2,

    # kernel arguments
    dt_min=0.001,
    dt_max=0.1,
    lr=0.001
)

In [5]:
data = LitBNSDataRegression(data_cfg=data_cfg)
model = LitModelS4DMSE(model_cfg=model_cfg)

In [6]:
trainer = L.Trainer(
    logger=CSVLogger('../outputs', 'train101'),
    # profiler='simple',
    fast_dev_run=True,
    callbacks=[DeviceStatsMonitor(cpu_stats=True)]
)
trainer.fit(model=model, datamodule=data)

/n/home04/kyoon/miniforge3/envs/ssm_cuda312/lib/python3.12/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /n/home04/kyoon/miniforge3/envs/ssm_cuda312/lib/pyth ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
Running in `fast_dev_run` mode: will run the requested loop using 1 batch(es). Logging and checkpointing is suppressed.

  | Name      | Type            | Params | Mode  | FLOPs
--------------------------------------------------------------
0 | criterion | MSELoss         | 0      | train | 0    
1 | model     | OptimizedModule | 37     | train | 0    
----

Epoch 0:   0%|          | 0/1 [00:00<?, ?it/s]

/n/home04/kyoon/miniforge3/envs/ssm_cuda312/lib/python3.12/site-packages/torch/_inductor/lowering.py:1713: UserWarning: Torchinductor does not support code generation for complex operators. Performance may be worse than eager.
  warnings.warn(


Epoch 0: 100%|██████████| 1/1 [00:09<00:00,  0.11it/s, val/loss=1.250, train/loss=1.250]

`Trainer.fit` stopped: `max_steps=1` reached.


Epoch 0: 100%|██████████| 1/1 [00:09<00:00,  0.11it/s, val/loss=1.250, train/loss=1.250]


In [7]:
# Please provide ckpt_path in the arguments
# trainer.test(datamodule=data)
# trainer.validate(datamodule=data)
# trainer.predict(datamodule=data)